# 描述性统计

## 导入库

In [1]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
dotenv.load_dotenv()

True

## 超参数

In [3]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'baseline1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "connectorx"



## 读取数据  
1.FM回归的控制变量  
2.MA数据

In [26]:
fm_reg = pl.read_database_uri(
    """SELECT * 
    FROM statics.fm_reg_controls""",
    uri = CONNECTION_URL,
    engine = ENGINE
)


In [27]:
fm_describe = fm_reg.select(pl.all().exclude(['stkcd','accper'])).describe()

In [28]:
ma_describe = pl.read_parquet(f"{SAVE_BASE_DIR}/MA因子_copy.parquet").select(pl.col(['MA', 'return'])).describe()

In [29]:
ma_describe

statistic,MA,return
str,f64,f64
"""count""",402640.0,402640.0
"""null_count""",0.0,0.0
"""mean""",0.669343,0.004048
"""std""",0.113507,0.135024
"""min""",0.054467,-0.836998
"""25%""",0.597326,-0.065788
"""50%""",0.682496,-0.003415
"""75%""",0.752675,0.0614
"""max""",1.0,4.153096


In [30]:
desc =ma_describe.join(fm_describe, on = 'statistic', how = 'left')

In [ ]:
# 先 unpivot：统计量保留为 index，各变量列压成 variable + value
long = desc.unpivot(
    index="statistic",
    on=[c for c in desc.columns if c != "statistic"],  # 或直接写 ["MA", "return", "betavals", ...]
    variable_name="variable",
    value_name="value"
)

# 再 pivot：行=变量，列=统计量
result = long.pivot(
    values="value",
    index="variable",
    on = "statistic"
)

result

statistic,MA,return,betavals,bm_ratio,gross_margin,investment_ratio,ln_market_value
str,f64,f64,f64,f64,f64,f64,f64
"""count""",402640.0,402640.0,998924.0,325753.0,318602.0,402584.0,999501.0
"""null_count""",0.0,0.0,1076.0,674247.0,681398.0,597416.0,499.0
"""mean""",0.669343,0.004048,1.11061,0.650698,0.265466,0.942977,15.06076
"""std""",0.113507,0.135024,5.73821,0.264081,3.826124,0.103233,1.320145
"""min""",0.054467,-0.836998,-2376.66667,-0.221668,-2137.8044,-0.310563,9.939355
"""25%""",0.597326,-0.065788,0.72842,0.464737,0.1505,0.93515,14.187714
"""50%""",0.682496,-0.003415,1.10061,0.65581,0.2446,0.989542,14.984791
"""75%""",0.752675,0.0614,1.4775,0.83641,0.3714,0.999999,15.818205
"""max""",1.0,4.153096,2540.83333,43.587682,2.1605,1.23341,21.747961


In [37]:
result.write_parquet(f"/home/frank/files/programs/GraduationThesis/empirical/描述性统计.parquet")

## 附录-因子信息

In [5]:
factors = pl.read_database_uri(
    """SELECT * 
    FROM factors_data.factor_metadata""",
    uri = CONNECTION_URL,
    engine = ENGINE
)


In [ ]:
cates = factors['category'].unique()

category
str
"""财务流动性因子"""
"""动量因子"""
"""交易摩擦因子"""
"""价值因子"""
"""成长因子"""
"""盈利因子"""


In [ ]:
for cate in cates